# Etapa 4: Avaliação e Casos de Teste (Golden Set)

Conforme exigido pelo Datathon, vamos avaliar as métricas finais do modelo e criar um "Golden Set" de 5 clientes para demonstrar qual seria o **próximo passo recomendado** (Prioridade de Contato) para cada um deles baseado no que nosso Algoritmo Adaptativo (Thompson Sampling) aprendeu na Etapa 3.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/bank_cleaned.csv')

## 1. O que o Algoritmo Aprendeu?
Na Etapa 3, o Thompson Sampling aprendeu que as taxas de conversão variam muito por segmento (profissão). As maiores taxas observadas empiricamente (e aprendidas pelas distribuições Beta do algoritmo) são dos segmentos `student` e `retired`.

Vamos simular a política de recomendação gerada pelo algoritmo:
- Se o cliente pertence a um segmento com alta conversão histórica (ex: > 15%), o próximo passo é **Contato Prioritário**.
- Se o cliente pertence a um segmento mediano (ex: entre 10% e 15%), o próximo passo é **Contato Padrão**.
- Se o cliente pertence a um segmento de baixa conversão (ex: < 10%), o próximo passo é **Baixa Prioridade / Não Contatar**.

In [2]:
# Taxas aprendidas (usando as médias empíricas como proxy do que as distribuições Beta convergiram)
taxas_aprendidas = df.groupby('job')['target'].mean()

def recomendar_proximo_passo(job):
    taxa = taxas_aprendidas.get(job, 0)
    if taxa > 0.15:
        return "Contato Prioritário (Alta propensão)"
    elif taxa >= 0.10:
        return "Contato Padrão (Média propensão)"
    else:
        return "Baixa Prioridade (Baixa propensão)"

## 2. Golden Set (5 Exemplos de Clientes)
Vamos selecionar 5 clientes de perfis diferentes e ver o que a nossa plataforma recomendaria para eles.

In [3]:
# Selecionando 5 clientes com profissões variadas para o Golden Set
profissoes_teste = ['student', 'blue-collar', 'admin.', 'retired', 'services']
golden_set = df[df['job'].isin(profissoes_teste)].drop_duplicates(subset=['job']).head(5)

# Gerando as recomendações
recomendacoes = []
for index, row in golden_set.iterrows():
    recomendacao = recomendar_proximo_passo(row['job'])
    recomendacoes.append({
        'ID Cliente': index,
        'Idade': row['age'],
        'Profissão (Segmento)': row['job'],
        'Estado Civil': row['marital'],
        'Recomendação do Algoritmo': recomendacao
    })

golden_df = pd.DataFrame(recomendacoes)
display(golden_df)

,ID Cliente,Idade,Profissão (Segmento),Estado Civil,Recomendação do Algoritmo
0,1,57,services,married,Baixa Prioridade (Baixa propensão)
1,3,40,admin.,married,Contato Padrão (Média propensão)
2,7,41,blue-collar,married,Baixa Prioridade (Baixa propensão)
3,15,54,retired,married,Contato Prioritário (Alta propensão)
4,205,35,student,single,Contato Prioritário (Alta propensão)


## 3. Análise das Decisões (Fez sentido?)

1. **Estudantes (student) e Aposentados (retired):** O algoritmo recomendou **Contato Prioritário**. Faz total sentido, pois esses foram os segmentos que o Bandit mais explorou e descobriu que dão a maior taxa de retorno. O custo de oportunidade de não ligar para eles é alto.
2. **Admin (admin.):** O algoritmo recomendou **Contato Padrão**. É um segmento mediano, então recebe a régua de comunicação normal.
3. **Trabalhadores Manuais (blue-collar) e Serviços (services):** O algoritmo classificou como **Baixa Prioridade**. Faz sentido, pois ao longo da exploração do Bandit, esses braços retornaram muitas falhas (target=0), então o algoritmo aprendeu a evitar gastar esforço aqui.

Isso conclui a avaliação e a explicabilidade das decisões do modelo para os públicos de negócio!